# Lab Day 2: Backbone, công thức huấn luyện và suy luận trên DeepWeeds (Colab)

**Chạy trên Google Colab, GPU T4 trở lên** (Runtime → Change runtime type → GPU). Mọi kết quả (runs, checkpoint, predictions, curves)
ghi thẳng vào **Google Drive** nên phiên bị ngắt thì chỉ cần chạy lại ô 1–3 rồi chạy tiếp: `train.run` tự bỏ qua thí nghiệm đã xong
(so khớp `config.json`).

Thứ tự: **Bước 0** (EDA, kiểm tra pipeline) → **Bước 1** (6 backbone) → **Bước 2** (công thức huấn luyện) → **Bước 3** (suy luận, độ trễ)
→ **Bước 4** (chung kết ≥ 3 seed, test một lần mỗi seed) → **Bước 5** (xlsx, báo cáo).

**Quy tắc vàng:** mọi lựa chọn dựa trên **val**. Test chỉ được chạm ở Bước 4, qua `final.finalize()` (có chốt chặn: mỗi seed một lần).
Các ô có chữ **`# <<< BẠN ĐIỀN`** là chỗ bạn quyết định sau khi nhìn kết quả val.

## 0. Cài đặt (chạy lại sau mỗi lần Colab ngắt)

In [ ]:
import os, sys, platform, subprocess
from google.colab import drive
drive.mount("/content/drive")

WORK = "/content/drive/MyDrive/lab_day2_deepweeds"          # nơi lưu bền: runs/, predictions/, curves/, results.xlsx
os.makedirs(WORK, exist_ok=True)

REPO_URL = "https://github.com/duclongt23/K4-Track4-Day2-Deeplearning-Advance.git"
REPO = "/content/K4-Track4-Day2-Deeplearning-Advance"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)   # lấy code mới nhất bạn đã push

CODE = f"{REPO}/submissions/2A202602917_NguyenDucLong/code"
sys.path.insert(0, CODE)
os.environ["EVAL_DIR"] = REPO                                  # để train.import_eval() tìm eval.py
!pip -q install timm thop openpyxl seaborn

import numpy as np, pandas as pd, torch, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU -> đổi runtime!")

### Tải dữ liệu vào đĩa cục bộ của Colab (nhanh hơn đọc từ Drive)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và fold chia sẵn từ GitHub tác giả. **Không sửa các file CSV.**

In [ ]:
import hashlib, glob
DATA = "/content/data"
os.makedirs(f"{DATA}/labels", exist_ok=True)
if not os.path.exists(f"{DATA}/images.zip"):
    !wget -q -O {DATA}/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
h = hashlib.md5()
with open(f"{DATA}/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
if not glob.glob(f"{DATA}/**/20160928-140314-0.jpg", recursive=True):
    !unzip -q -n {DATA}/images.zip -d {DATA}/
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{DATA}/labels/{name}.csv"):
        !wget -q -O {DATA}/labels/{name}.csv {BASE}/{name}.csv
IMAGES_DIR = os.path.dirname(glob.glob(f"{DATA}/**/20160928-140314-0.jpg", recursive=True)[0])
LABELS_DIR = f"{DATA}/labels"
print("IMAGES_DIR =", IMAGES_DIR, "| số jpg:", len(glob.glob(f"{IMAGES_DIR}/*.jpg")))

In [ ]:
EPOCHS = 10          # GIỐNG NHAU cho mọi backbone/thí nghiệm (GUIDE 1.4: 10-15). Đổi thì phải chạy lại tất cả.
COMMON = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{WORK}/runs",
              pred_dir=f"{WORK}/predictions", curves_dir=f"{WORK}/curves",
              epochs=EPOCHS, batch_size=64, num_workers=2, amp=True)
SEEDS = (0, 1, 2)    # seed cho vòng chung kết và đo nhiễu
from dataclasses import replace
import dataset, losses, model as M, train, experiments, inference, inference_suite, final, build_results
from train import Config, run

## Bước 0: EDA và kiểm tra pipeline
Bắt buộc đưa vào báo cáo: số ảnh mỗi tập/lớp, giao rỗng, hợp 17.509, biểu đồ phân bố lớp, ảnh mẫu.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, 0)
stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)          # assert: 60/20/20, giao rỗng, đủ 17509, đủ file
cnt = pd.DataFrame({k: pd.Series(v) for k, v in stats["per_class"].items()})
cnt.index = dataset.CLASS_NAMES; cnt.loc["Tổng"] = cnt.sum()
display(cnt)
print("Tỉ lệ lớp lớn nhất / nhỏ nhất (train):", cnt.iloc[:-1]["train"].max() / cnt.iloc[:-1]["train"].min())
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for a, (n, d) in zip(ax, [("train", train_df), ("val", val_df), ("test", test_df)]):
    sns.countplot(x=d["Label"].map(lambda i: dataset.CLASS_NAMES[i]), ax=a, order=dataset.CLASS_NAMES); a.set_title(n)
    a.tick_params(axis="x", rotation=70)
plt.tight_layout(); os.makedirs(f"{WORK}/figs", exist_ok=True); plt.savefig(f"{WORK}/figs/eda_class_distribution.png", dpi=130); plt.show()

fig, ax = plt.subplots(9, 3, figsize=(7, 20))
for l in range(9):
    for j, fn in enumerate(train_df[train_df.Label == l].Filename.head(3)):
        ax[l, j].imshow(Image.open(f"{IMAGES_DIR}/{fn}")); ax[l, j].axis("off")
    ax[l, 0].set_title(dataset.CLASS_NAMES[l], fontsize=9, loc="left")
plt.tight_layout(); plt.savefig(f"{WORK}/figs/eda_samples.png", dpi=110); plt.show()
im = Image.open(f"{IMAGES_DIR}/{train_df.Filename[0]}"); print("kích thước ảnh:", im.size, im.mode)

In [ ]:
# Kiểm tra pipeline (GUIDE 1.3): loss ban đầu ~ ln 9, overfit 1 batch nhỏ, nhìn ảnh sau augmentation
import math
train.set_seed(0)
dev = "cuda"
m = M.build_model("resnet50", True, 9).to(dev)
tf = dataset.build_transforms(True, 224, "basic")
x, y, _ = next(iter(dataset.make_loader(train_df.head(16), IMAGES_DIR, tf, 16, False, None, 0)))
x, y = x.to(dev), y.to(dev)
m.eval()
with torch.no_grad():
    print(f"loss ban đầu = {torch.nn.functional.cross_entropy(m(x), y).item():.3f} (kỳ vọng ~ ln 9 = {math.log(9):.3f})")
opt = torch.optim.AdamW(m.parameters(), lr=1e-4); m.train()
for i in range(40):
    opt.zero_grad(); l = torch.nn.functional.cross_entropy(m(x), y); l.backward(); opt.step()
m.eval()
with torch.no_grad():
    print(f"loss sau khi overfit 1 batch 16 ảnh = {torch.nn.functional.cross_entropy(m(x), y).item():.4f} (kỳ vọng ~ 0)")
del m, opt
mean, std = torch.tensor(dataset.IMAGENET_MEAN).view(3, 1, 1), torch.tensor(dataset.IMAGENET_STD).view(3, 1, 1)
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for a, i in zip(ax.ravel(), range(12)):
    a.imshow((x[i].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)); a.set_title(dataset.CLASS_NAMES[y[i].item()], fontsize=8); a.axis("off")
plt.suptitle("Sau augmentation (đã giải chuẩn hoá): ảnh và nhãn phải khớp"); plt.savefig(f"{WORK}/figs/aug_check.png", dpi=110); plt.show()

## Bước 1: So sánh 6 backbone (B01–B06)
Cùng công thức nền, cùng split, cùng seed 0. Mỗi lần ghi params, GMAC, macro-F1/top-1 val, thời gian/epoch, độ trễ sơ bộ, **tag trọng số**.
Ước lượng: chạy B01 trước, xem thời gian mỗi epoch, rồi nhân ra tổng ngân sách (GUIDE mục 7).

In [ ]:
res_b = [run(c) for c in experiments.backbone_cfgs(COMMON, seed=0)]
df_b = build_results.sheet_backbones(build_results.collect_runs(COMMON["out_dir"]))
display(df_b)

## TỰ ĐỘNG: chạy một ô là xong Bước 2 -> 4 (thay cho các ô thủ công ở trên)
Chỉ cần đã chạy xong Bước 0-1. Quy tắc chọn cố định, chỉ dựa trên val (xem auto_pipeline.py); quyết định lưu ở auto_decisions.json.
Mất kết nối thì chạy lại 3 ô cài đặt + ô khai báo cấu hình rồi chạy lại ô này: phần đã xong được bỏ qua.
Xong ô này, chạy tiếp các ô `eval.py` và Bước 5 (chúng dùng các biến do ô này gán).

In [ ]:
import auto_pipeline
KEEP = None   # None = chạy đủ T01-T12 (6 trục). Muốn rút gọn: KEEP = {"T00","T01","T06","T07","T08","T12"}
AUTO = auto_pipeline.run_all(COMMON, "convnext_tiny", "deit_small_patch16_224", WORK, seeds=SEEDS, keep=KEEP)
BEST_CFG, F01, T00 = AUTO["BEST_CFG"], AUTO["F01"], AUTO["T00"]
FINAL_TTA, FINAL_TS, P95_REALTIME = AUTO["FINAL_TTA"], AUTO["FINAL_TS"], AUTO["P95_REALTIME"]
inf_df, lat_df = AUTO["inf_df"], AUTO["lat_df"]
BEST_BACKBONE = BEST_CFG.backbone
P, T = f"{WORK}/predictions", LABELS_DIR

In [ ]:
# Đường đánh đổi độ chính xác - độ trễ (batch 1, p50)
d = inf_df.dropna(subset=["p50_ms"])
plt.figure(figsize=(7, 5)); plt.scatter(d.p50_ms, d.macro_f1_val)
for _, r in d.iterrows(): plt.annotate(r.exp_id, (r.p50_ms, r.macro_f1_val), fontsize=8)
plt.xlabel("độ trễ p50 batch-1 (ms)"); plt.ylabel("macro-F1 val"); plt.grid(alpha=.3); plt.title("Đánh đổi chính xác - độ trễ (val)")
plt.savefig(f"{WORK}/figs/tradeoff.png", dpi=130); plt.show()

In [ ]:
P, T = f"{WORK}/predictions", LABELS_DIR
!python {REPO}/eval.py score --pred "{P}/F01_seed*_test.csv" --test-csv {T}/test_subset0.csv --labels {T}/labels.csv --tag F01 --out {WORK}/eval_out
!python {REPO}/eval.py score --pred "{P}/T00_seed*_test.csv" --test-csv {T}/test_subset0.csv --labels {T}/labels.csv --tag T00 --out {WORK}/eval_out

In [ ]:
# Tự chấm RUBRIC phần I. Điền p95 batch-1 (ms) của cấu hình dùng được cho thời gian thực lấy từ inf_df/Latency nếu có.
P95_REALTIME = None   # <<< BẠN ĐIỀN (ms), ví dụ giá trị p95_ms của dòng I08a hoặc backbone nhẹ; None nếu chưa có
extra = f"--latency-p95-ms {P95_REALTIME}" if P95_REALTIME else ""
import glob as _g
UNCAL = f'--uncal "{P}/F01uncal_seed*_test.csv"' if _g.glob(f"{P}/F01uncal_seed*_test.csv") else ""   # chỉ có khi đã dùng temperature scaling
!python {REPO}/eval.py grade --final "{P}/F01_seed*_test.csv" --baseline "{P}/T00_seed*_test.csv" \
    {UNCAL} --final-val "{P}/F01_seed*_val.csv" \
    --test-csv {T}/test_subset0.csv --val-csv {T}/val_subset0.csv --labels {T}/labels.csv --out {WORK}/eval_out {extra}

## Bước 5: Sản phẩm (results.xlsx, ma trận nhầm lẫn, phân tích lỗi)

In [ ]:
final_ids = {"T00": f"{BEST_BACKBONE} + công thức nền + 1 view (mốc)",
             "F01": f"{BEST_CFG.backbone} + {BEST_CFG.exp_id} + TTA={FINAL_TTA}, TS={FINAL_TS}"}
sheets = build_results.build_xlsx(f"{WORK}/results.xlsx", COMMON["out_dir"], inf_df, lat_df, P, T, final_ids)
display(sheets["Final"]); display(sheets["PerClass"].query("`cấu hình` == 'F01'")); display(sheets["Summary"])

In [ ]:
# Ma trận nhầm lẫn (test, F01, gộp các seed) + các ảnh bị đoán sai (phân tích lỗi: Chinee Apple <-> Snake Weed)
import glob as g
ev = train.import_eval()
preds = [ev.read_pred(f) for f in sorted(g.glob(f"{P}/F01_seed*_test.csv"))]
cm = sum(ev.confusion_matrix(p.y_true, p.y_pred) for p in preds)
plt.figure(figsize=(8, 7)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset.CLASS_NAMES, yticklabels=dataset.CLASS_NAMES)
plt.xlabel("dự đoán"); plt.ylabel("thật"); plt.title(f"F01 - ma trận nhầm lẫn (test, cộng {len(preds)} seed)"); plt.tight_layout()
plt.savefig(f"{WORK}/figs/confusion_F01_test.png", dpi=130); plt.show()
p0 = preds[0]; wrong = np.where(p0.y_true != p0.y_pred)[0]
pairs = [i for i in wrong if {p0.y_true[i], p0.y_pred[i]} == {0, 7}][:12] or list(wrong[:12])
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
for a, i in zip(ax.ravel(), pairs):
    a.imshow(Image.open(f"{IMAGES_DIR}/{p0.filenames[i]}")); a.axis("off")
    a.set_title(f"thật: {dataset.CLASS_NAMES[p0.y_true[i]]}\nđoán: {dataset.CLASS_NAMES[p0.y_pred[i]]}", fontsize=8)
for a in ax.ravel()[len(pairs):]: a.axis("off")
plt.savefig(f"{WORK}/figs/errors_F01.png", dpi=110); plt.show()

In [ ]:
# Gói sản phẩm nhỏ để đưa vào thư mục bài nộp trong repo (KHÔNG gồm runs/ chứa checkpoint)
import shutil
OUT = f"{WORK}/submission_bundle"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
for item in ["results.xlsx", "curves", "predictions", "figs", "eval_out"]:
    src = f"{WORK}/{item}"
    if os.path.isdir(src): shutil.copytree(src, f"{OUT}/{item}")
    elif os.path.exists(src): shutil.copy(src, OUT)
shutil.make_archive(f"{WORK}/submission_bundle", "zip", OUT)
print("Đã tạo", f"{WORK}/submission_bundle.zip", "-> tải về, giải nén vào submissions/2A202602917_NguyenDucLong/")

In [ ]:
# TỰ ĐỘNG KẾT THÚC: tải gói kết quả về máy rồi tắt Colab để khỏi tốn GPU. Kết quả vẫn nằm trên Drive.
import time
from google.colab import files, runtime
try:
    files.download(f"{WORK}/submission_bundle.zip")   # chỉ thành công nếu tab trình duyệt còn mở
    time.sleep(90)                                    # chờ trình duyệt nhận file
except Exception as e:
    print("Không tải được tự động (không sao, file có trên Drive):", e)
drive.flush_and_unmount()                             # đảm bảo mọi thứ đã ghi xuống Drive
runtime.unassign()                                    # ngắt và giải phóng runtime

In [ ]:
# Chọn backbone đi tiếp sang Bước 2-4 DỰA TRÊN SỐ LIỆU VAL (macro-F1, và đánh đổi độ trễ/tham số). Ghi lý do vào report.
BEST_BACKBONE = "resnet50"     # <<< BẠN ĐIỀN sau khi xem df_b
SECOND_BACKBONE = "convnext_tiny"  # <<< BẠN ĐIỀN: backbone thứ hai để thử ensemble ở Bước 3 (tuỳ chọn)

## Bước 2: Công thức huấn luyện (T00–T12)
Mỗi dòng khác `T00` đúng một yếu tố. Trục: A khởi tạo · B augmentation · C loss · D sampler · E LR · F EMA (đủ ≥ 3 trục, có loss và augmentation).
1 seed cho quét sàng; **nhiễu seed đo bằng T00 chạy 3 seed** (ô bên dưới) để biết Δ nào là thật.

In [ ]:
T_CFGS = experiments.training_cfgs(BEST_BACKBONE, COMMON, seed=0)
for cfg, axis, desc in T_CFGS:
    run(cfg)
runs = build_results.collect_runs(COMMON["out_dir"])

In [ ]:
# Nhiễu seed: T00 thêm seed 1, 2 (cũng chính là mốc cho chung kết, nên không tốn thêm)
T00 = T_CFGS[0][0]
for s in SEEDS[1:]:
    run(replace(T00, seed=s))
runs = build_results.collect_runs(COMMON["out_dir"])
t00 = runs[runs.exp_id == "T00"]
NOISE = t00.val_macro_f1.std(ddof=1)
print("T00 macro-F1 val theo seed:", dict(zip(t00.seed, t00.val_macro_f1.round(4))), "| std =", round(NOISE, 4))
df_t = build_results.sheet_training(runs, NOISE)
display(df_t)

In [ ]:
# Kết hợp các yếu tố THẮNG RÕ (|Δ| > NOISE) để xem cộng dồn hay triệt tiêu. Ghi rõ đây là kết hợp, và dùng cách tham lam theo trục.
WINNERS = ["T06", "T07"]       # <<< BẠN ĐIỀN, ví dụ ["T06", "T07", "T12"]
COMBO = experiments.combo_cfg(BEST_BACKBONE, COMMON, WINNERS, seed=0, exp_id="T13")
print(COMBO)
run(COMBO)
runs = build_results.collect_runs(COMMON["out_dir"])
display(build_results.sheet_training(runs, NOISE).tail(3))

## Bước 3: Suy luận trên VAL (I00–I08) và độ trễ
Không huấn luyện lại. Dùng checkpoint của **cấu hình tốt nhất ở Bước 2** (chọn `T13` hay `T00`... theo val). Độ trễ đo đúng cách:
warmup 10 lần, `cuda.synchronize`, ≥ 50 lần (mặc định 100), p50/p95/p99, batch 1 và batch 32.

In [ ]:
BEST_CFG = COMBO                 # <<< BẠN ĐIỀN: cấu hình tốt nhất trên val (COMBO hoặc một cfg trong T_CFGS)
SECOND_CFG = experiments.backbone_cfgs(COMMON)[[n for _, n, _ in experiments.BACKBONES].index(SECOND_BACKBONE)]  # thành viên ensemble (B-run seed 0)
inf_df, lat_df = inference_suite.run_suite(BEST_CFG, ensemble_cfgs=[SECOND_CFG], sizes=(224, 256, 288, 320), iters=100)
display(build_results.sheet_inference(inf_df))
inf_df.to_csv(f"{WORK}/inference_val.csv", index=False); lat_df.to_csv(f"{WORK}/latency.csv", index=False)

## Bước 4: Chung kết, ≥ 3 seed, **test đúng một lần mỗi seed**
1. Cấu hình chung kết = `BEST_CFG` (đổi `exp_id` thành `F01`) + phương pháp suy luận chốt ở Bước 3 trên val (`TTA`, `TEMPERATURE`).
2. Mốc = `T00` + `I00` (1 view, không hiệu chuẩn), cùng 3 seed.
3. `finalize` ghi `predictions/<exp_id>_seed<k>_{val,test}.csv` và bản `uncal`. Gọi lần hai sẽ báo lỗi (chốt chặn quy tắc S4).
> Sau khi mở kết quả test, **không** quay lại sửa cấu hình rồi chạy test lại.

In [ ]:
FINAL_TTA = "none"          # <<< BẠN ĐIỀN: "none" hoặc "hflip" theo kết quả I01/I03 trên VAL
FINAL_TS = True             # <<< BẠN ĐIỀN: temperature scaling (T khớp trên val) theo I07
F01 = replace(BEST_CFG, exp_id="F01")
for s in SEEDS:
    cfg_f = replace(F01, seed=s, save_val_predictions=False)
    run(cfg_f)                                                     # train (bỏ qua nếu đã xong)
    final.finalize(cfg_f, tta=FINAL_TTA, temperature=FINAL_TS)     # TEST đúng một lần
    cfg_b = replace(T00, seed=s)
    run(cfg_b)                                                     # đã có từ Bước 2
    final.finalize(cfg_b, tta="none", temperature=False)           # mốc T00 + I00

---
# PHỤ LỤC: các ô chạy THỦ CÔNG (không cần nếu dùng ô TỰ ĐỘNG; Run all sẽ không tới đây vì runtime đã tắt)